# Speech Emotion Recognition — Transformer Model
**Course: ITE 5410 Deep Learning | Dataset: CREMA-D**

This single notebook covers the full pipeline:
1. Google Colab setup
2. Data preparation (parse labels, train/val/test split)
3. Feature extraction (MFCC sequences)
4. Transformer Encoder model
5. Training with evaluation
6. Results & visualizations

| Item | Detail |
|------|--------|
| Dataset | CREMA-D — 7,442 WAV files |
| Emotions | ANG, DIS, FEA, HAP, NEU, SAD (6 classes) |
| Input | MFCC sequences (200 frames × 40 coefficients) |
| Model | 4-layer Transformer Encoder with Positional Encoding |
| Framework | PyTorch |

---
## STEP 0 — Google Colab Setup
Run these cells first. Skip if running locally.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install required libraries not pre-installed on Colab
!pip install librosa tqdm -q

In [ ]:
# ============================================================
#  SET YOUR PROJECT PATH HERE
#  This should be the folder on Google Drive where you
#  uploaded the Crema/ dataset folder.
# ============================================================
BASE_DIR = '/content/drive/MyDrive/SER_Project'

# Verify the path exists and show what's inside
import os
print('BASE_DIR exists:', os.path.exists(BASE_DIR))
print('Contents:', os.listdir(BASE_DIR))

---
## STEP 1 — Import Libraries

In [ ]:
import os
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, f1_score
)
from tqdm import tqdm

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch  : {torch.__version__}')
print(f'Device   : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU      : {torch.cuda.get_device_name(0)}')

---
## STEP 2 — Configuration
All hyperparameters and paths in one place.

In [ ]:
# Paths
CREMA_DIR    = os.path.join(BASE_DIR, 'Crema')
FEATURES_DIR = os.path.join(BASE_DIR, 'features')
os.makedirs(FEATURES_DIR, exist_ok=True)

# Emotion labels
EMOTION_MAP  = {'ANG': 0, 'DIS': 1, 'FEA': 2, 'HAP': 3, 'NEU': 4, 'SAD': 5}
EMOTION_FULL = ['Anger', 'Disgust', 'Fear', 'Happiness', 'Neutral', 'Sadness']
NUM_CLASSES  = 6

# Audio settings
SAMPLE_RATE  = 22050
DURATION     = 3.0       # seconds — clips trimmed/padded to this length
N_MFCC       = 40        # MFCC coefficients per frame
MAX_FRAMES   = 200       # fixed sequence length (time steps)

# Transformer hyperparameters
D_MODEL      = 128       # embedding dimension
NHEAD        = 8         # attention heads  (D_MODEL must be divisible by NHEAD)
NUM_LAYERS   = 4         # transformer encoder layers
DIM_FF       = 256       # feed-forward hidden size
DROPOUT      = 0.2

# Training
BATCH_SIZE   = 64
EPOCHS       = 60
LR           = 5e-4

print('Configuration set.')
print(f'CREMA-D path : {CREMA_DIR}')
print(f'Features path: {FEATURES_DIR}')

---
## STEP 3 — Data Preparation
Parse all 7,442 filenames and create a stratified 70/15/15 split.

File naming convention: `[ActorID]_[Sentence]_[Emotion]_[Intensity].wav`  
Example: `1001_DFA_ANG_XX.wav` → Anger

In [ ]:
records = []
for fname in sorted(os.listdir(CREMA_DIR)):
    if not fname.endswith('.wav'):
        continue
    parts = fname.replace('.wav', '').split('_')
    if len(parts) != 4:
        continue
    actor_id, sentence, emotion_code, intensity = parts
    if emotion_code not in EMOTION_MAP:
        continue
    records.append({
        'filepath'     : os.path.join(CREMA_DIR, fname),
        'filename'     : fname,
        'actor_id'     : int(actor_id),
        'emotion_code' : emotion_code,
        'emotion_label': EMOTION_MAP[emotion_code],
        'emotion_name' : EMOTION_FULL[EMOTION_MAP[emotion_code]],
    })

df = pd.DataFrame(records)
print(f'Total files parsed: {len(df)}')
print()
print('Class distribution:')
print(df['emotion_name'].value_counts().to_string())

In [ ]:
# Stratified split: 70% train / 15% val / 15% test
df_train, df_temp = train_test_split(
    df, test_size=0.30, random_state=SEED, stratify=df['emotion_label']
)
df_val, df_test = train_test_split(
    df_temp, test_size=0.50, random_state=SEED, stratify=df_temp['emotion_label']
)
df_train = df_train.copy(); df_train['split'] = 'train'
df_val   = df_val.copy();   df_val['split']   = 'val'
df_test  = df_test.copy();  df_test['split']  = 'test'

print(f'Train : {len(df_train):,} ({len(df_train)/len(df)*100:.1f}%)')
print(f'Val   : {len(df_val):,} ({len(df_val)/len(df)*100:.1f}%)')
print(f'Test  : {len(df_test):,} ({len(df_test)/len(df)*100:.1f}%)')

In [ ]:
# Visualize class distribution
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
colors = ['#e74c3c','#8e44ad','#2980b9','#27ae60','#95a5a6','#e67e22']

for ax, (subset, title) in zip(axes, [
    (df_train, 'Train'), (df_val, 'Validation'), (df_test, 'Test')
]):
    counts = subset['emotion_name'].value_counts()
    ax.bar(counts.index, counts.values, color=colors)
    ax.set_title(f'{title} ({len(subset)} samples)')
    ax.set_xlabel('Emotion')
    ax.tick_params(axis='x', rotation=20)
    for i, v in enumerate(counts.values):
        ax.text(i, v + 5, str(v), ha='center', fontsize=9)

axes[0].set_ylabel('Count')
plt.suptitle('Class Distribution Across Splits', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print('Stratified split confirmed — class balance maintained.')

---
## STEP 4 — Feature Extraction (MFCC Sequences)

Each WAV file → MFCC matrix of shape **(200 time frames × 40 coefficients)**

- Audio is resampled to 22,050 Hz and padded/trimmed to 3 seconds
- 40 MFCC coefficients are extracted per frame
- Sequences shorter than 200 frames are zero-padded; longer ones are trimmed
- Features are saved as `.npy` files so this step can be skipped on restart

In [ ]:
def load_audio(filepath):
    """Load WAV, resample, pad/trim to DURATION seconds."""
    y, _ = librosa.load(filepath, sr=SAMPLE_RATE, duration=DURATION)
    target = int(SAMPLE_RATE * DURATION)
    if len(y) < target:
        y = np.pad(y, (0, target - len(y)))
    return y[:target]


def extract_mfcc_seq(y):
    """Extract MFCC matrix → (MAX_FRAMES, N_MFCC)."""
    mfcc = librosa.feature.mfcc(y=y, sr=SAMPLE_RATE, n_mfcc=N_MFCC).T  # (T, 40)
    if mfcc.shape[0] < MAX_FRAMES:
        pad = np.zeros((MAX_FRAMES - mfcc.shape[0], N_MFCC))
        mfcc = np.vstack([mfcc, pad])
    return mfcc[:MAX_FRAMES].astype(np.float32)  # (200, 40)


def extract_and_save(split_df, split_name):
    feat_path  = os.path.join(FEATURES_DIR, f'mfcc_{split_name}.npy')
    label_path = os.path.join(FEATURES_DIR, f'labels_{split_name}.npy')

    if os.path.exists(feat_path) and os.path.exists(label_path):
        print(f'[{split_name}] Already extracted — loading from cache.')
        return np.load(feat_path), np.load(label_path)

    n = len(split_df)
    X = np.zeros((n, MAX_FRAMES, N_MFCC), dtype=np.float32)
    y = split_df['emotion_label'].values
    errors = 0

    for i, (_, row) in enumerate(tqdm(split_df.iterrows(), total=n, desc=f'Extracting [{split_name}]')):
        try:
            audio = load_audio(row['filepath'])
            X[i]  = extract_mfcc_seq(audio)
        except Exception as e:
            errors += 1
            print(f'  Error: {row["filename"]} — {e}')

    np.save(feat_path,  X)
    np.save(label_path, y)
    print(f'[{split_name}] Saved: X={X.shape}, errors={errors}')
    return X, y

print('Functions defined. Starting extraction...')

In [ ]:
# This cell takes ~5-8 minutes on Colab (7,442 files)
# If your session resets, just re-run — it loads from cache automatically
X_train, y_train = extract_and_save(df_train, 'train')
X_val,   y_val   = extract_and_save(df_val,   'val')
X_test,  y_test  = extract_and_save(df_test,  'test')

print()
print(f'Train shape : {X_train.shape}  — (samples, frames, mfcc_coeffs)')
print(f'Val shape   : {X_val.shape}')
print(f'Test shape  : {X_test.shape}')

In [ ]:
# Normalize: zero-mean, unit-variance using train statistics only
X_train_t = torch.FloatTensor(X_train)
X_val_t   = torch.FloatTensor(X_val)
X_test_t  = torch.FloatTensor(X_test)

mean = X_train_t.mean(dim=(0, 1), keepdim=True)  # (1, 1, 40)
std  = X_train_t.std(dim=(0, 1),  keepdim=True) + 1e-8

X_train_t = (X_train_t - mean) / std
X_val_t   = (X_val_t   - mean) / std
X_test_t  = (X_test_t  - mean) / std

y_train_t = torch.LongTensor(y_train)
y_val_t   = torch.LongTensor(y_val)
y_test_t  = torch.LongTensor(y_test)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=BATCH_SIZE, shuffle=True,  pin_memory=True)
val_loader   = DataLoader(TensorDataset(X_val_t,   y_val_t),   batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)
test_loader  = DataLoader(TensorDataset(X_test_t,  y_test_t),  batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)

print('Data normalized and loaders ready.')

In [ ]:
# Visualize a sample MFCC sequence
sample_idx   = 0
sample_mfcc  = X_train[sample_idx]   # (200, 40)
sample_label = EMOTION_FULL[y_train[sample_idx]]

plt.figure(figsize=(14, 4))
plt.imshow(sample_mfcc.T, aspect='auto', origin='lower', cmap='viridis')
plt.colorbar(label='MFCC Value')
plt.title(f'Sample MFCC Sequence — Emotion: {sample_label}', fontsize=13, fontweight='bold')
plt.xlabel('Time Frame (0–200)')
plt.ylabel('MFCC Coefficient (0–39)')
plt.tight_layout()
plt.show()

---
## STEP 5 — Transformer Model

### Architecture
```
Input: (batch, 200, 40)  — MFCC sequence
  ↓
Linear Projection: (batch, 200, 40) → (batch, 200, 128)   [d_model]
  ↓
Positional Encoding (sinusoidal — adds position info)
  ↓
TransformerEncoderLayer × 4  (8 attention heads, FFN=256)
  ↓
Mean Pooling over time:  (batch, 200, 128) → (batch, 128)
  ↓
LayerNorm → Linear(128→64) → GELU → Dropout → Linear(64→6)
  ↓
Output: (batch, 6)  — emotion logits
```

In [ ]:
class PositionalEncoding(nn.Module):
    """
    Injects position information into the sequence using fixed sinusoidal signals.
    Without this, the Transformer has no notion of which frame comes first.
    """
    def __init__(self, d_model, max_len=500, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe       = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])


class SpeechEmotionTransformer(nn.Module):
    def __init__(self, input_size=40, d_model=128, nhead=8,
                 num_layers=4, dim_ff=256, dropout=0.2, num_classes=6):
        super().__init__()

        # Project 40-dim MFCC → d_model (128)
        self.input_proj = nn.Linear(input_size, d_model)

        self.pos_encoding = PositionalEncoding(d_model, dropout=dropout)

        # Transformer Encoder: 4 layers, 8 attention heads each
        encoder_layer = nn.TransformerEncoderLayer(
            d_model         = d_model,
            nhead           = nhead,
            dim_feedforward = dim_ff,
            dropout         = dropout,
            batch_first     = True,   # (batch, seq, features)
            norm_first      = True    # Pre-LN: more stable training
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Classification head
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        # x: (batch, 200, 40)
        x = self.input_proj(x)    # → (batch, 200, 128)
        x = self.pos_encoding(x)  # → (batch, 200, 128) + positional info
        x = self.encoder(x)       # → (batch, 200, 128) — self-attention
        x = x.mean(dim=1)         # mean pooling → (batch, 128)
        return self.classifier(x) # → (batch, 6)


model = SpeechEmotionTransformer(
    input_size = N_MFCC,
    d_model    = D_MODEL,
    nhead      = NHEAD,
    num_layers = NUM_LAYERS,
    dim_ff     = DIM_FF,
    dropout    = DROPOUT,
    num_classes= NUM_CLASSES
).to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f'\nTotal parameters    : {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

---
## STEP 6 — Training

In [ ]:
# Loss, optimizer, scheduler
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)   # label smoothing reduces overconfidence
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=20, T_mult=1, eta_min=1e-5
)

# History
train_losses, val_losses = [], []
train_accs,   val_accs   = [], []
lr_history = []
best_val_acc  = 0.0
WEIGHTS_PATH  = os.path.join(BASE_DIR, 'best_transformer.pth')

print(f'Training for {EPOCHS} epochs on {DEVICE}...')
print(f'Best model will be saved to: {WEIGHTS_PATH}')
print('-' * 70)

In [ ]:
for epoch in range(1, EPOCHS + 1):

    # ---- Train ----
    model.train()
    t_loss, t_correct, t_total = 0, 0, 0
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
        optimizer.zero_grad()
        logits = model(X_b)
        loss   = criterion(logits, y_b)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # prevents exploding gradients
        optimizer.step()
        t_loss    += loss.item() * len(y_b)
        t_correct += (logits.argmax(1) == y_b).sum().item()
        t_total   += len(y_b)

    train_losses.append(t_loss / t_total)
    train_accs.append(t_correct / t_total)

    # ---- Validate ----
    model.eval()
    v_loss, v_correct, v_total = 0, 0, 0
    val_criterion = nn.CrossEntropyLoss()  # no label smoothing for validation
    with torch.no_grad():
        for X_b, y_b in val_loader:
            X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
            logits  = model(X_b)
            loss    = val_criterion(logits, y_b)
            v_loss    += loss.item() * len(y_b)
            v_correct += (logits.argmax(1) == y_b).sum().item()
            v_total   += len(y_b)

    val_losses.append(v_loss / v_total)
    val_accs.append(v_correct / v_total)
    lr_history.append(optimizer.param_groups[0]['lr'])
    scheduler.step()

    # Save best model
    if val_accs[-1] > best_val_acc:
        best_val_acc = val_accs[-1]
        torch.save(model.state_dict(), WEIGHTS_PATH)

    if epoch % 10 == 0:
        print(f'Epoch {epoch:3d}/{EPOCHS} | '
              f'Train Loss: {train_losses[-1]:.4f}, Acc: {train_accs[-1]*100:.2f}% | '
              f'Val Loss: {val_losses[-1]:.4f}, Acc: {val_accs[-1]*100:.2f}% | '
              f'LR: {lr_history[-1]:.6f}')

print()
print(f'Training complete. Best Val Accuracy: {best_val_acc*100:.2f}%')
print(f'Weights saved to: {WEIGHTS_PATH}')

---
## STEP 7 — Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(train_losses, label='Train Loss', color='steelblue')
axes[0].plot(val_losses,   label='Val Loss',   color='coral')
axes[0].set_title('Loss Curve', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.4)

# Accuracy
axes[1].plot([a*100 for a in train_accs], label='Train Acc', color='steelblue')
axes[1].plot([a*100 for a in val_accs],   label='Val Acc',   color='coral')
axes[1].set_title('Accuracy Curve', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
axes[1].legend(); axes[1].grid(True, alpha=0.4)

# Learning rate
axes[2].plot(lr_history, color='green')
axes[2].set_title('Learning Rate Schedule', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('LR')
axes[2].grid(True, alpha=0.4)

plt.suptitle('Transformer Training — Speech Emotion Recognition', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'transformer_training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Training curves saved.')

---
## STEP 8 — Evaluation on Test Set

In [ ]:
# Load best saved weights
model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=DEVICE))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for X_b, y_b in test_loader:
        preds = model(X_b.to(DEVICE)).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y_b.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

test_acc = accuracy_score(all_labels, all_preds)
test_f1  = f1_score(all_labels, all_preds, average='weighted')

print('=' * 55)
print('  TEST SET RESULTS — Transformer Encoder')
print('=' * 55)
print(f'  Accuracy : {test_acc*100:.2f}%')
print(f'  F1 Score : {test_f1*100:.2f}%  (weighted)')
print('=' * 55)
print()
print(classification_report(all_labels, all_preds, target_names=EMOTION_FULL))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(9, 7))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='YlOrRd',
    xticklabels=EMOTION_FULL, yticklabels=EMOTION_FULL,
    linewidths=0.5
)
plt.title(
    f'Confusion Matrix — Transformer Encoder\nTest Accuracy: {test_acc*100:.2f}%  |  F1: {test_f1*100:.2f}%',
    fontsize=13, fontweight='bold'
)
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'transformer_confusion_matrix.png'), dpi=150)
plt.show()

In [ ]:
# Per-emotion F1 score
f1_per_class = f1_score(all_labels, all_preds, average=None)
colors = ['#e74c3c','#8e44ad','#2980b9','#27ae60','#95a5a6','#e67e22']

plt.figure(figsize=(9, 5))
bars = plt.bar(EMOTION_FULL, f1_per_class * 100, color=colors, alpha=0.87, edgecolor='white')
for bar, val in zip(bars, f1_per_class):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{val*100:.1f}%', ha='center', fontsize=11, fontweight='bold')
plt.title('Per-Emotion F1 Score — Transformer Encoder', fontsize=13, fontweight='bold')
plt.ylabel('F1 Score (%)')
plt.ylim(0, 108)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'transformer_per_emotion_f1.png'), dpi=150)
plt.show()

---
## STEP 9 — Final Summary

In [ ]:
print('=' * 60)
print('  SPEECH EMOTION RECOGNITION — FINAL SUMMARY')
print('=' * 60)
print(f'  Dataset       : CREMA-D ({len(df):,} WAV files)')
print(f'  Emotion classes: 6 (ANG, DIS, FEA, HAP, NEU, SAD)')
print(f'  Model         : Transformer Encoder')
print(f'  Input         : MFCC sequences ({MAX_FRAMES} frames × {N_MFCC} coefficients)')
print(f'  Architecture  : {NUM_LAYERS} encoder layers, {NHEAD} attention heads, d_model={D_MODEL}')
print(f'  Parameters    : {sum(p.numel() for p in model.parameters()):,}')
print()
print(f'  Best Val Acc  : {best_val_acc*100:.2f}%')
print(f'  Test Accuracy : {test_acc*100:.2f}%')
print(f'  Test F1 Score : {test_f1*100:.2f}%  (weighted average)')
print()
print('  Per-Emotion Results:')
for i, emotion in enumerate(EMOTION_FULL):
    print(f'    {emotion:<12}: F1 = {f1_per_class[i]*100:.1f}%')
print()
print(f'  Saved files:')
print(f'    - {WEIGHTS_PATH}')
print(f'    - {BASE_DIR}/transformer_training_curves.png')
print(f'    - {BASE_DIR}/transformer_confusion_matrix.png')
print(f'    - {BASE_DIR}/transformer_per_emotion_f1.png')
print('=' * 60)